# Swish with a reusable device function

Swish (SiLU when $\beta = 1$) is the activation inside the SwiGLU blocks of Llama-style models:

$$\mathrm{swish}_\beta(x) = x\,\sigma(\beta x), \qquad \sigma(z) = \frac{1}{1+e^{-z}}$$

Kernels are easier to read and test when shared maths lives in **device functions**: functions decorated with `@cuda.jit(device=True)` that kernels call like normal Python functions. They are inlined at compile time, so they cost nothing.

## Task
Write `swish(x, beta)`:
- `x`: 2-D `float32` array of shape `(rows, cols)`, either dimension possibly 0. `beta`: Python float (any sign).
- Return a `float32` array of the same shape.
- Put an overflow-safe sigmoid (never call `math.exp` on a positive argument) in a device function `stable_sigmoid(z)` and call it from the kernel.
- Launch a 1-D kernel over all `rows * cols` elements of the flattened array, 128 threads per block, no launch when the array is empty.

Inputs may be as large as $\pm 500$.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import math
import numpy as np
from numba import cuda

TPB = 128

def swish(x, beta):
    """x: float32 (rows, cols), beta: float. Returns x * sigmoid(beta * x), same shape, float32."""
    flat = np.ascontiguousarray(x).ravel()
    if flat.size == 0:
        return flat.reshape(x.shape)
    d_x = cuda.to_device(flat)
    d_out = cuda.device_array(flat.size, dtype=np.float32)
    swish_kernel[(flat.size + TPB - 1) // TPB, TPB](d_x, beta, d_out)
    return d_out.copy_to_host().reshape(x.shape)

@cuda.jit(device=True)
def stable_sigmoid(z):
    # TODO: overflow-safe sigmoid
    return 0.5

@cuda.jit
def swish_kernel(x, beta, out):
    i = cuda.grid(1)
    if i < x.shape[0]:
        out[i] = x[i] * stable_sigmoid(beta * x[i])


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[-1.0, 0.0, 1.0], [2.0, -2.0, 3.0]], "dtype": "float32"}, 1.0], "e": {"__nd__": [[-0.2689414322376251, 0.0, 0.7310585975646973], [1.7615941762924194, -0.23840583860874176, 2.857722282409668]], "dtype": "float32"}}, {"a": [{"__nd__": [[-1.0, 0.0, 1.0], [2.0, -2.0, 3.0]], "dtype": "float32"}, 2.0], "e": {"__nd__": [[-0.11920291930437088, 0.0, 0.8807970881462097], [1.9640275239944458, -0.03597242012619972, 2.992582082748413]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.0]], "dtype": "float32"}, 1.0], "e": {"__nd__": [[0.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[-500.0, 500.0], [-100.0, 100.0]], "dtype": "float32"}, 1.0], "e": {"__nd__": [[-0.0, 500.0], [-3.783505853677006e-42, 100.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[3.0, -3.0]], "dtype": "float32"}, -1.0], "e": {"__nd__": [[0.14227762818336487, -2.857722282409668]], "dtype": "float32"}}, {"a": [{"__nd__": [[3.0, -3.0, 0.5]], "dtype": "float32"}, 0.0], "e": {"__nd__": [[1.5, -1.5, 0.25]], "dtype": "float32"}}, {"a": [{"__nd__": [[-6.953065395355225, -5.34657096862793, -5.444427013397217, -1.4064685106277466, -9.250326156616211, -0.7555887699127197, -3.8289170265197754, 3.5744006633758545, 3.8273890018463135, 5.569033145904541, 3.069880485534668, -0.21211911737918854, 3.439175844192505, 6.0219244956970215, -2.6143782138824463, 2.4414045810699463, -0.17069530487060547, 5.760066986083984, -3.3475801944732666, -1.2061864137649536], [1.4493544101715088, 1.0324410200119019, -6.557791709899902, 1.4406208992004395, -0.473990797996521, -0.9589914083480835, -0.6212066411972046, 0.8758867979049683, -7.26558256149292, 6.209866046905518, -3.4457666873931885, -8.965471267700195, -0.32789796590805054, 5.829921722412109, -2.074404001235962, 6.205102443695068, 6.2277679443359375, -3.450927734375, -9.860483169555664, -4.940731048583984], [4.749728679656982, -3.2670886516571045, -6.042709827423096, -5.350778579711914, 0.000720576907042414, -0.10443393141031265, 3.4881796836853027, 3.956199884414673, -3.7289960384368896, -0.6270347833633423, -4.535738468170166, 0.28756269812583923, -4.602431297302246, -4.801006317138672, 8.493687629699707, 0.12955555319786072, 2.5727145671844482, 10.152393341064453, 3.143355369567871, -0.45657041668891907], [0.2183598130941391, -2.9779679775238037, 4.377353668212891, -1.3211147785186768, 7.0509819984436035, -1.379034161567688, -1.4144917726516724, -1.6775095462799072, 2.5027754306793213, 0.7477442622184753, 5.485729694366455, -2.2133984565734863, 1.1609525680541992, 7.751846790313721, -0.9615267515182495, 2.130451202392578, 6.287528991699219, 2.213783025741577, -2.538020610809326, -8.24940013885498], [9.572654724121094, 1.7212570905685425, 5.042115688323975, -1.9052542448043823, 7.273926734924316, 2.078157424926758, 0.19918976724147797, -3.607268810272217, 1.4727388620376587, -1.9461201429367065, -0.8910767436027527, -6.4900431632995605, -1.9622156620025635, 0.6395161151885986, -5.003224849700928, 0.2955860495567322, 2.6531755924224854, -1.2270398139953613, -2.4831366539001465, -2.480496644973755], [3.7645692825317383, -6.138020038604736, -1.201805591583252, -1.6776204109191895, 0.5759881734848022, -0.2782803475856781, -4.43002462387085, 8.160357475280762, 0.7104470729827881, -5.368153095245361, -3.5657339096069336, 0.3903394937515259, 2.4518797397613525, -5.3686203956604, 2.8453474044799805, -5.493168354034424, -3.1775553226470947, 3.412405014038086, 6.803988933563232, -0.33883148431777954], [-1.238154411315918, 5.071596622467041, -4.2879157066345215, 1.1116873025894165, 0.9329822659492493, 4.03728723526001, 1.1943137645721436, -1.0655773878097534, 2.4582183361053467, -4.506259441375732, 3.256305694580078, -2.6907060146331787, -3.6414635181427, -1.470585584640503, 0.1840057671070099, 2.84234356880188, -0.8506826758384705, 2.9428913593292236, 3.900329828262329, -0.42303523421287537], [-3.9261884689331055, 3.679004430770874, 1.1017929315567017, -8.460774421691895, -3.9531545639038086, 0.5726467370986938, -5.487622261047363, -2.7256805896759033, -6.851688385009766, 1.609054446220398, 0.9660712480545044, 0.7045024037361145, 5.789896488189697, -1.4104067087173462, -1.156720757484436, 8.079692840576172, -5.203452110290527, 2.916651725769043, -5.260385990142822, 0.9450231790542603], [8.463130950927734, 1.0854240655899048, -3.0872910022735596, 9.536864280700684, -2.175285577774048, -0.8247860074043274, 1.2174882888793945, -5.370260715484619, 4.460392951965332, 6.475300312042236, 3.0350632667541504, 1.3214441537857056, -0.5971907377243042, 3.5469422340393066, -12.600313186645508, -0.4990810453891754, 1.0245028734207153, -7.942120552062988, 1.7046693563461304, -0.5100199580192566], [-0.13826388120651245, 0.40556493401527405, 2.3005480766296387, 1.8318501710891724, -4.149407863616943, -0.46784791350364685, -0.7283933162689209, 0.15368975698947906, -2.3643651008605957, 3.87941575050354, 5.218193531036377, 6.086799144744873, 7.6226091384887695, 5.973122596740723, -1.8397265672683716, -2.457507371902466, -1.5905557870864868, 2.2337541580200195, -0.5193225741386414, 3.964374303817749], [6.601565837860107, -1.7210450172424316, 3.9188308715820312, -1.5717791318893433, -5.065989971160889, -5.075374126434326, -1.2765185832977295, -7.5964035987854, 1.9972419738769531, 6.042724132537842, -4.660248279571533, -1.3372563123703003, 6.174233436584473, 1.388927936553955, 0.7826665043830872, -1.0461359024047852, 4.074635028839111, 1.3169692754745483, -4.3367509841918945, -9.728373527526855], [-10.72676944732666, 2.630685806274414, -1.064337968826294, 5.7925190925598145, 2.03997802734375, -5.08814811706543, 1.1615557670593262, 4.767446517944336, -3.6070268154144287, -2.0349695682525635, 6.963948726654053, -6.985568046569824, -4.361147880554199, 4.200159549713135, 0.7959829568862915, -4.569963455200195, -5.820981502532959, 1.8538190126419067, 4.8570780754089355, 3.30633807182312]], "dtype": "float32"}, 1.702], "e": {"__nd__": [[-5.0429083785275e-05, -0.0005970436614006758, -0.000514704966917634, -0.11764544248580933, -1.34462277401326e-06, -0.16360656917095184, -0.005652189254760742, 3.566270112991333, 3.8217244148254395, 5.568607330322266, 3.0534508228302, -0.08711966872215271, 3.4293339252471924, 6.021711349487305, -0.030189529061317444, 2.4037106037139893, -0.07303638011217117, 5.759748935699463, -0.011190473102033138, -0.13721179962158203], [1.33598792552948, 0.8805280923843384, -9.320380922872573e-05, 1.3263839483261108, -0.14626751840114594, -0.15682266652584076, -0.16016381978988647, 0.7148919701576233, -3.0958039133111015e-05, 6.2097063064575195, -0.009751028381288052, -2.116277300956426e-06, -0.11935200542211533, 5.8296356201171875, -0.059023045003414154, 6.204941749572754, 6.2276129722595215, -0.009680469520390034, -5.07378786096524e-07, -0.0011006725253537297], [4.748264312744141, -0.012518785893917084, -0.00020636308181565255, -0.0005932503845542669, 0.00036050938069820404, -0.04758845269680023, 3.478994369506836, 3.951495885848999, -0.0065233767963945866, -0.16047915816307068, -0.0020126900635659695, 0.17828097939491272, -0.0018232223810628057, -0.0013566045090556145, 8.493682861328125, 0.07189084589481354, 2.5408501625061035, 10.152392387390137, 3.1285006999969482, -0.14379636943340302], [0.1292378455400467, -0.018619749695062637, 4.374810695648193, -0.12613436579704285, 7.050938606262207, -0.12038315087556839, -0.11684440821409225, -0.0912853553891182, 2.4679131507873535, 0.5841362476348877, 5.485246658325195, -0.050010181963443756, 1.0196040868759155, 7.751832485198975, -0.15667054057121277, 2.0752058029174805, 6.287387371063232, 2.1637961864471436, -0.03332182392477989, -6.587478310393635e-06], [9.572653770446777, 1.6339712142944336, 5.041170597076416, -0.07161573320627213, 7.273896217346191, 2.0193934440612793, 0.11631739884614944, -0.007759989704936743, 1.3616992235183716, -0.0684095174074173, -0.16035893559455872, -0.00010351430682931095, -0.06717520952224731, 0.47841620445251465, -0.0010021512862294912, 0.18420468270778656, 2.62447452545166, -0.13525323569774628, -0.035747427493333817, -0.03586790710687637], [3.7583699226379395, -0.00017822982044890523, -0.1376194804906845, -0.09127510339021683, 0.41884365677833557, -0.10679193586111069, -0.002353084273636341, 8.16034984588623, 0.547153651714325, -0.0005778360646218061, -0.008231226354837418, 0.25771722197532654, 2.4146828651428223, -0.0005774269229732454, 2.823086977005005, -0.00047797380830161273, -0.014171011745929718, 3.4021859169006348, 6.803925037384033, -0.12187575548887253], [-0.13419806957244873, 5.070692539215088, -0.002900394145399332, 0.9660494327545166, 0.7746789455413818, 4.03310489654541, 1.0560003519058228, -0.149396151304245, 2.4213194847106934, -0.0021024474408477545, 3.243597984313965, -0.02732413075864315, -0.007391553837805986, -0.11125357449054718, 0.10629291087388992, 2.819993734359741, -0.16191114485263824, 2.9233665466308594, 3.895230293273926, -0.1384982019662857], [-0.0049125803634524345, 3.6719980239868164, 0.9553258419036865, -4.714829174190527e-06, -0.004724700469523668, 0.4157669246196747, -0.00048201941535808146, -0.026095030829310417, -5.905231228098273e-05, 1.511331558227539, 0.8096767067909241, 0.5413100719451904, 5.789592266082764, -0.11725225299596786, -0.1417265385389328, 8.079684257507324, -0.0007413069834001362, 2.896423101425171, -0.0006802144926041365, 0.787386953830719], [8.463126182556152, 0.9376116394996643, -0.016042953357100487, 9.536863327026367, -0.052363645285367966, -0.16266103088855743, 1.0813343524932861, -0.0005759934429079294, 4.4581427574157715, 6.475194454193115, 3.017833709716797, 1.1953423023223877, -0.15868885815143585, 3.5384888648986816, -6.118111350872368e-09, -0.14950032532215118, 0.872012734413147, -1.069953304977389e-05, 1.6158782243728638, -0.150792196393013], [-0.06103502959012985, 0.27011731266975403, 2.255594253540039, 1.7542189359664917, -0.0035522186663001776, -0.14541763067245483, -0.1635124832391739, 0.086838498711586, -0.04152911528944969, 3.874160051345825, 5.217468738555908, 6.086606025695801, 7.622591495513916, 5.972892761230469, -0.07697027921676636, -0.03693215921521187, -0.09949591755867004, 2.184965133666992, -0.1518358737230301, 3.9597256183624268], [6.6014790534973145, -0.08730505406856537, 3.913865804672241, -0.1013084426522255, -0.0009119313326664269, -0.0008991474751383066, -0.13050483167171478, -1.8432290744385682e-05, 1.93269681930542, 6.04251766204834, -0.0016731737414374948, -0.12453783303499222, 6.174065113067627, 1.2695317268371582, 0.6192359924316406, -0.15089310705661774, 4.070673942565918, 1.190425992012024, -0.0026996112428605556, -6.267958951866603e-07], [-1.2634882295969874e-07, 2.601130247116089, -0.1494932323694229, 5.7922163009643555, 1.9785374402999878, -0.0008820269722491503, 1.020261287689209, 4.76602029800415, -0.007762659806758165, -0.061798498034477234, 6.9638991355896, -4.7938228817656636e-05, -0.002604444744065404, 4.196861267089844, 0.6327325701713562, -0.0019131740555167198, -0.00028992409352213144, 1.7780221700668335, 4.855830669403076, 3.2944846153259277]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(swish, json.loads(_TESTS), 'cuda', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")